# Hyperparameter Tuning + Feature Engineering: Grid Search, Pipelines, and Leakage Detection

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/09_tuning_feature_engineering_project_baseline_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Run `GridSearchCV` and `RandomizedSearchCV` on known models and read `cv_results_` as a table of NB08-style CV runs
2. Apply the 95% confidence-interval overlap rule from NB08 to pick the simplest model among the top candidates in `cv_results_`
3. Build a `ColumnTransformer` that handles both categorical and numeric features inside a single `Pipeline`
4. Use `FunctionTransformer` to embed domain-specific feature engineering (ratios, bins) inside the pipeline without leakage
5. Detect a data-leakage bug in a provided pipeline by comparing CV scores before and after the fix
6. Explain why every feature-engineering step must live inside the pipeline that `cross_val_score` or `GridSearchCV` evaluates

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**, one at the end of each big section. Please complete them before submitting your notebook.

---

## 💼 Why This Matters: From One Knob to a Dial (and a Leaky Pipeline to Fix)

Two business cases on today's desk — one returning, one brand new.

**Section A — HomeValue Analytics + MedScreen.** In NB05 the CFO approved a Ridge baseline on California Housing with a single hand-picked $\alpha=1.0$. After seeing NB08's confidence intervals, the board wants a *defensible* choice: sweep a whole grid of $\alpha$ values, pick the best by 5-fold CV, and show that the choice survives the CI-overlap test from NB08. The Health Department wants the same treatment for MedScreen's logistic regression — a principled $C$-grid search instead of one hand-chosen value. The tool for both is `GridSearchCV`.

**Section B — TechCorp Talent Analytics.** You have been loaned to **TechCorp**, a 1,500-person SaaS company. The People Analytics team wants to flag employees at high risk of resigning in the next 6 months so HR can start retention conversations early. The economics are sharp: losing a mid-level engineer costs the company roughly \$75,000 (recruitment fees + 6 months of productivity loss + ramp time); a retention conversation costs about \$500 (manager time + modest retention bonus). A modest improvement over "managers guessing" pays for itself many times over.

TechCorp's dataset has something California Housing and Breast Cancer never had: **real categorical columns** — `department`, `job_level`, `remote_status`. That means you finally need `ColumnTransformer` to mix one-hot encoding for the categoricals with standardization for the numerics. And while you were out, an intern shipped a first draft of the modeling pipeline. Their CV ROC-AUC looked suspiciously good… because it is. You will find the leak, fix it, watch the inflated score collapse, and then find a second, subtler leak on your own.

By the end of today you own a workflow that plugs directly into your Kaggle competition submission: **ColumnTransformer → FunctionTransformer → GridSearchCV → `cv_results_` ranked by CI overlap → one final pipeline refit on all training data**.

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, GridSearchCV, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

print('Setup complete.')
print(f'Random seed: {RANDOM_SEED}')

**Reading the output:**

The setup cell imports today's full toolbox:

- **`GridSearchCV` / `RandomizedSearchCV`** — the two tuners. Grid is exhaustive on a small grid; randomized samples from distributions when the grid is too big.
- **`ColumnTransformer`** — lets one preprocessing pipeline apply different transforms to different columns (e.g., `StandardScaler` on numerics, `OneHotEncoder` on categoricals).
- **`OneHotEncoder`** — turns `department = ['Eng','Sales',…]` into binary indicator columns. Setting `handle_unknown='ignore'` means a category that only appears at prediction time won't crash the pipeline.
- **`FunctionTransformer`** — wraps a plain Python function into a pipeline step so a domain formula (e.g., *salary divided by market median*) refits correctly on every CV fold.
- **`SelectKBest`** — picks the top-$k$ features by a univariate score. It is leakage-prone if placed outside the pipeline, which is exactly the trap Section B uses.
- **`scipy.stats`** — for the same Student's $t$ CI computation from NB08. The CI-overlap rule drives both exercises today.

`RANDOM_SEED = 474` is the same seed used in NB01–NB08, so every split, every fold, and every CV score you see today is reproducible.

---

## 1. Section A — Grid Search as NB08 × a Grid

### 1.1 From one CV run to many

In NB08 you ran 5-fold CV on *one* pipeline (Ridge $\alpha=1.0$) and got one mean, one SD, one 95% CI. That answered: *is my single validation score representative?*

Today's question is one level up: *which hyperparameter value is best?* To answer it, run NB08's CV ritual on every candidate and compare the resulting CIs.

`GridSearchCV` does exactly that. Given a pipeline and a dict `{'param_name': [value1, value2, ...]}`, it runs 5-fold CV for *each* combination of values, stores every fold's score, and builds a `cv_results_` table where each row is one configuration. Think of the table as "a stack of NB08 runs."

The professional workflow:

1. Rank `cv_results_` by `mean_test_score`.
2. Read off the **top 3–5 rows** and compute each one's 95% CI.
3. If the top row's CI overlaps the second row's CI, the two are statistically tied — pick the **simpler** model (stronger regularization on a Ridge grid; smaller $C$ on a LogReg grid).
4. Refit the chosen pipeline on all training data and lock it until final evaluation.

This is the protocol the rest of the course will use.

---

### 1.2 GridSearchCV on Ridge — HomeValue Analytics

Load California Housing, apply the same 60/20/20 split used in NB01–NB08, and sweep Ridge over a 6-value $\alpha$ grid.

> 💡 **Gemini Prompt:** "Load the California Housing dataset. Do a 60/20/20 train/val/test split with `random_state=RANDOM_SEED`. Build a pipeline with `StandardScaler` + `Ridge`. Run `GridSearchCV` with `param_grid={'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}`, `cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)`, `scoring='r2'`, and `return_train_score=False`. After fitting, print `grid.best_params_` and `grid.best_score_`. Then build a DataFrame from `grid.cv_results_` with columns `param_ridge__alpha`, `mean_test_score`, `std_test_score`, `rank_test_score`, sort by rank, and print the full table."
>
> **After running, verify:**
> - The grid fits 6 configurations × 5 folds = 30 sub-fits
> - `best_params_` is printed with the winning $\alpha$
> - The `cv_results_` DataFrame has 6 rows, one per $\alpha$
> - `rank_test_score` starts at 1 for the best row


In [ ]:
# --- Load California Housing and apply the 60/20/20 split ---
cal = fetch_california_housing(as_frame=True)
X_reg = cal.data
y_reg = cal.target

X_reg_temp, X_reg_test, y_reg_temp, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_reg_train, X_reg_val, y_reg_train, y_reg_val = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.25, random_state=RANDOM_SEED
)
print(f'Train: {len(X_reg_train)} | Val: {len(X_reg_val)} | Test: {len(X_reg_test)} (locked)')

# --- Build pipeline + grid ---
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=RANDOM_SEED))
])

alpha_grid = {'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

grid_ridge = GridSearchCV(
    ridge_pipeline,
    param_grid=alpha_grid,
    cv=cv_reg,
    scoring='r2',
    return_train_score=False,
    n_jobs=-1
)
grid_ridge.fit(X_reg_train, y_reg_train)

print(f'\nBest params: {grid_ridge.best_params_}')
print(f'Best 5-fold CV R^2 (mean): {grid_ridge.best_score_:.4f}')

# --- cv_results_ as a ranked table ---
results_ridge = (
    pd.DataFrame(grid_ridge.cv_results_)
    [['param_ridge__alpha', 'mean_test_score', 'std_test_score', 'rank_test_score']]
    .sort_values('rank_test_score')
    .reset_index(drop=True)
)
print('\nRanked grid results:')
print(results_ridge)

**Reading the output:**

`grid.best_params_` prints the single winner — the $\alpha$ with the highest mean 5-fold $R^2$. `grid.best_score_` is that mean. The `cv_results_` table ranks every $\alpha$ tried, with `std_test_score` giving the fold-to-fold spread you already know how to convert into a 95% CI.

**A single number is not a decision.** The mean tells you who wins a footrace, not by how much. In the next cell you add the CI-overlap rule from NB08: if the top row's CI overlaps the second row's, the two are statistically tied and you should pick the simpler (more regularized) model.

---

### 1.3 CI-overlap rule on `cv_results_`

`GridSearchCV` stores every fold's score in columns like `split0_test_score`, `split1_test_score`, …, `split4_test_score`. Compute each row's 95% CI the same way as NB08 (Student's $t$, 4 degrees of freedom, SD with `ddof=1`) and visualize it.

In [ ]:
# --- Compute per-row 95% CI from fold scores ---
k = 5
t_crit = stats.t.ppf(0.975, df=k - 1)

fold_cols = [f'split{i}_test_score' for i in range(k)]
cvr = pd.DataFrame(grid_ridge.cv_results_).copy()
cvr['mean'] = cvr[fold_cols].mean(axis=1)
cvr['sd'] = cvr[fold_cols].std(axis=1, ddof=1)
cvr['half_w'] = t_crit * cvr['sd'] / np.sqrt(k)
cvr['ci_low'] = cvr['mean'] - cvr['half_w']
cvr['ci_high'] = cvr['mean'] + cvr['half_w']
cvr = cvr.sort_values('mean', ascending=False).reset_index(drop=True)

print('Top candidates by mean 5-fold CV R^2 (with 95% CI):\n')
print(cvr[['param_ridge__alpha', 'mean', 'sd', 'ci_low', 'ci_high']].head())

# --- Plot top 6 with CI error bars ---
fig, ax = plt.subplots(figsize=(10, 5))
labels = [f"α={a}" for a in cvr['param_ridge__alpha']]
ax.bar(labels, cvr['mean'], yerr=cvr['half_w'], color='steelblue',
       capsize=8, edgecolor='black')
ax.set_ylabel('5-fold CV R^2')
ax.set_title('Ridge α grid — mean CV R^2 with 95% CI (ranked)',
             fontsize=12, fontweight='bold')
for i, (m, h) in enumerate(zip(cvr['mean'], cvr['half_w'])):
    ax.text(i, m + h + 0.003, f'{m:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

# --- CI-overlap between top-1 and top-2 ---
top1, top2 = cvr.iloc[0], cvr.iloc[1]
overlap = not (top1['ci_high'] < top2['ci_low'] or top2['ci_high'] < top1['ci_low'])
if overlap:
    print(f"\n→ Top-1 (α={top1['param_ridge__alpha']}) and Top-2 (α={top2['param_ridge__alpha']}) 95% CIs OVERLAP.")
    simpler = max(top1['param_ridge__alpha'], top2['param_ridge__alpha'])
    print(f'  Statistical tie — pick the simpler (larger α): α={simpler}.')
else:
    print(f"\n→ Top-1 (α={top1['param_ridge__alpha']}) CI does NOT overlap Top-2. "
          f"Top-1 wins outright.")

**Reading the output:**

Every bar is one NB08 run — its mean is the point estimate, the whiskers are the 95% CI. Bars whose CIs overlap are *statistically indistinguishable* on this dataset. When that happens at the top of the ranking, the professional move is to pick the simpler candidate (larger $\alpha$ = stronger regularization = fewer effective degrees of freedom).

This is also the answer to a common objection: *"Why doesn't `GridSearchCV` just pick the model with the best mean for me?"* It does — via `grid.best_params_`. But that's the best *mean*, not the best *supported* choice. Best-mean can flip between runs if you reshuffle the seed; a CI-overlap winner is stable.

---

### 1.4 RandomizedSearchCV — when the grid is too big

A grid of 4 `C` values × 3 `penalty` × 2 `solver` is already 24 fits × 5 folds = 120 sub-fits. Realistic grids for gradient boosting or neural nets balloon into the thousands. `RandomizedSearchCV` samples `n_iter` points from distributions instead of enumerating every combination — same output format (`cv_results_`, `best_params_`, `best_score_`), a fraction of the compute.

> 💡 **Gemini Prompt:** "On Breast Cancer (same stratified 60/20/20 split as NB06/NB07/NB08), build a pipeline `StandardScaler + LogisticRegression(max_iter=5000)`. Use `RandomizedSearchCV` with `param_distributions={'clf__C': scipy.stats.loguniform(1e-3, 1e3)}`, `n_iter=20`, `cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED)`, `scoring='roc_auc'`, `random_state=RANDOM_SEED`. Print `best_params_` and `best_score_`, then show the top 5 rows of `cv_results_` sorted by `rank_test_score`."
>
> **After running, verify:**
> - 20 random `C` values were sampled (each on a log scale)
> - `best_score_` is a mean ROC-AUC between 0.98 and 1.00
> - Top-5 rows show a range of `C` values, not all clustered in one spot

In [ ]:
# --- Breast Cancer stratified 60/20/20 ---
bc = load_breast_cancer(as_frame=True)
X_clf = bc.data
y_clf = bc.target

X_clf_temp, X_clf_test, y_clf_temp, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
X_clf_train, X_clf_val, y_clf_train, y_clf_val = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.25,
    random_state=RANDOM_SEED, stratify=y_clf_temp
)
print(f'Train: {len(X_clf_train)} | Val: {len(X_clf_val)} | Test: {len(X_clf_test)} (locked)')

# --- Randomized search over C ---
log_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])

rand_search = RandomizedSearchCV(
    log_pipeline,
    param_distributions={'clf__C': stats.loguniform(1e-3, 1e3)},
    n_iter=20,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
    scoring='roc_auc',
    random_state=RANDOM_SEED,
    n_jobs=-1
)
rand_search.fit(X_clf_train, y_clf_train)

print(f'\nBest params: {rand_search.best_params_}')
print(f'Best 5-fold CV ROC-AUC (mean): {rand_search.best_score_:.4f}')

# --- Top 5 ---
top5 = (
    pd.DataFrame(rand_search.cv_results_)
    [['param_clf__C', 'mean_test_score', 'std_test_score', 'rank_test_score']]
    .sort_values('rank_test_score')
    .head()
    .reset_index(drop=True)
)
print('\nTop 5 randomized-search candidates:')
print(top5)

**Reading the output:**

`RandomizedSearchCV` drew 20 `C` values from a log-uniform distribution between $10^{-3}$ and $10^{3}$. Printing the top 5 rows shows that the winning `C` is rarely a round number like $1.0$ — it tends to be something weirder like $C \approx 0.23$ or $C \approx 47$, which a fixed grid would never have tried.

**Important mental model:** every row of `cv_results_`, whether from `GridSearchCV` or `RandomizedSearchCV`, is one complete NB08 run. The tuner is just a convenient loop. When you quote a score from the winner, quote it as *"5-fold CV ROC-AUC = 0.99 ± CI"*, never as a single magic number.

---

## 📝 PAUSE-AND-DO Exercise 1 (10 minutes)

**Task:** Run `GridSearchCV` on MedScreen's Logistic Regression over an explicit `C` grid and apply the CI-overlap rule to pick a simpler winner than `best_params_` alone.

Build the same `StandardScaler + LogisticRegression` pipeline used in Section 1.4 (not `RandomizedSearchCV` this time — use `GridSearchCV` with the explicit grid below). Fit on `X_clf_train`/`y_clf_train`, using the same `StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED)` splitter and `scoring='roc_auc'`.

Grid:

```python
param_grid = {'clf__C': [0.01, 0.1, 1.0, 10.0, 100.0]}
```

After fitting, compute the 95% CI on every row's `splitN_test_score` columns, sort by mean, and check whether the top-1 CI overlaps the top-2 CI. If it does, pick the *smaller* `C` (stronger regularization = simpler model) as your champion; if it doesn't, the top-1 wins outright. Print a one-sentence verdict.

**Decision rule reminder (from NB08 and Section 1.3):** overlapping CIs at the top of the ranking are a statistical tie. Break ties toward the simpler model, not the one with the higher mean by 0.001.

---

> 💡 **Gemini Prompt:** "Using the `StandardScaler + LogisticRegression(random_state=RANDOM_SEED, max_iter=5000)` pipeline from Section 1.4, run `GridSearchCV` on `X_clf_train`, `y_clf_train` with `param_grid={'clf__C': [0.01, 0.1, 1.0, 10.0, 100.0]}`, `cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED)`, and `scoring='roc_auc'`. After fitting, build a DataFrame from `cv_results_` including the `split0_test_score` through `split4_test_score` columns. Compute `mean`, `sd` (ddof=1), `half_w = scipy.stats.t.ppf(0.975, 4) * sd / sqrt(5)`, `ci_low`, `ci_high` for every row, and sort by `mean` descending. Print the top-1 vs top-2 overlap check and the final verdict: `best_C_by_mean` vs `champion_by_CI` (the simpler one when CIs overlap)."
>
> **After running, verify:**
> - DataFrame has 5 rows, one per `C`, each with mean, sd, ci_low, ci_high
> - A printed line states whether the top-1 and top-2 CIs overlap
> - A final line prints `champion_by_CI = <value>` where `<value>` is either top-1's `C` (if no overlap) or the simpler `C` among the two (if overlap)

In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance.
#
# What to do:
#   1. Build pipeline: StandardScaler + LogisticRegression(random_state=RANDOM_SEED, max_iter=5000)
#   2. Run GridSearchCV with param_grid={'clf__C': [0.01, 0.1, 1.0, 10.0, 100.0]}
#      on X_clf_train, y_clf_train using StratifiedKFold(5, shuffle, random_state=RANDOM_SEED)
#      with scoring='roc_auc'
#   3. Extract cv_results_ as a DataFrame
#   4. For every row, compute mean / sd (ddof=1) / half_w / ci_low / ci_high across split0..split4
#   5. Sort by mean (descending)
#   6. Check whether top-1 and top-2 CIs overlap. Print the verdict.
#   7. champion_by_CI = simpler C when they overlap, otherwise top-1 C


### YOUR ANALYSIS:

**Question 1:** Which `C` has the highest mean 5-fold CV ROC-AUC?  
[Your answer]

**Question 2:** Do the top-1 and top-2 95% CIs overlap? What do you conclude?  
[Your answer]

**Question 3:** Which `C` do you recommend to the Health Department, and why? Reference the CI-overlap rule in your justification.  
[Your answer]

---

## 2. Section B — Feature Engineering, Categorical Data, and the Leakage Trap

### 2.1 TechCorp Talent Analytics — the business case

**TechCorp** is a 1,500-person SaaS company. The People Analytics team is under a standing request from the CHRO: *flag employees at high risk of resigning within 6 months so HR can run targeted retention conversations*. The economics are unambiguous — losing a mid-level engineer costs roughly \$75,000 (recruitment fees + 6 months of productivity loss + ramp time); a retention conversation costs about \$500 (manager time + a modest retention bonus). Even a modest lift over manager intuition pays for itself many times over.

The People Analytics team assembled a dataset with:

- **Numeric features:** `years_at_company`, `satisfaction_score` (1–10 pulse survey), `salary_pct_of_market` (company salary ÷ market median × 100), `projects_completed_last_year`, `manager_interactions_last_quarter`
- **Categorical features:** `department` (`Engineering`, `Sales`, `Marketing`, `Operations`, `Support`), `job_level` (`IC1`, `IC2`, `IC3`, `Manager`, `Director`), `remote_status` (`onsite`, `hybrid`, `remote`), `manager_id` (80 distinct managers — one per ~25 employees)
- **Target:** `left_within_6mo` (binary — 1 if the employee resigned within 6 months of the snapshot, else 0)

This is the first dataset in the course with real categorical columns. California Housing and Breast Cancer are both entirely numeric — so `ColumnTransformer` has been on the syllabus since NB02 but you have never seen it exercised on live categorical data. Today you do.

> 💡 **Gemini Prompt:** "Generate a synthetic TechCorp attrition dataset with 2000 rows. Use `numpy.random` seeded at `RANDOM_SEED`. Build a pandas DataFrame with: numeric columns `years_at_company` (integer 0–15), `satisfaction_score` (1–10), `salary_pct_of_market` (60–140), `projects_completed_last_year` (0–12), `manager_interactions_last_quarter` (0–20); categorical columns `department` (one of Engineering/Sales/Marketing/Operations/Support), `job_level` (IC1/IC2/IC3/Manager/Director), `remote_status` (onsite/hybrid/remote). Create a binary target `left_within_6mo` whose probability is higher when `satisfaction_score` is low, `salary_pct_of_market` is low, `manager_interactions_last_quarter` is low, `remote_status == 'onsite'`, or `department == 'Support'`. Target around 22% positive rate. Print the head, the class balance, and the dtypes so the numeric vs categorical split is obvious."


In [ ]:
# --- Generate synthetic TechCorp attrition dataset ---
rng = np.random.default_rng(RANDOM_SEED)
n = 2000

years = rng.integers(0, 16, size=n)
satisfaction = rng.integers(1, 11, size=n)
salary_pct = rng.uniform(60, 140, size=n).round(1)
projects = rng.integers(0, 13, size=n)
mgr_interactions = rng.integers(0, 21, size=n)

department = rng.choice(
    ['Engineering', 'Sales', 'Marketing', 'Operations', 'Support'],
    size=n, p=[0.35, 0.20, 0.15, 0.15, 0.15]
)
job_level = rng.choice(
    ['IC1', 'IC2', 'IC3', 'Manager', 'Director'],
    size=n, p=[0.25, 0.30, 0.25, 0.15, 0.05]
)
remote_status = rng.choice(
    ['onsite', 'hybrid', 'remote'],
    size=n, p=[0.30, 0.45, 0.25]
)

# --- High-cardinality categorical: 80 manager IDs (one manager per ~25 employees) ---
manager_id = rng.choice([f'MGR_{i:03d}' for i in range(80)], size=n)

# --- Attrition propensity ---
logit = (
    -2.0
    + 0.35 * (10 - satisfaction)
    + 0.025 * (100 - salary_pct)
    + 0.08 * (10 - mgr_interactions)
    + 0.40 * (remote_status == 'onsite').astype(float)
    + 0.35 * (department == 'Support').astype(float)
    - 0.05 * years
)
prob_leave = 1.0 / (1.0 + np.exp(-logit))
left_within_6mo = (rng.uniform(0, 1, size=n) < prob_leave).astype(int)

techcorp = pd.DataFrame({
    'years_at_company': years,
    'satisfaction_score': satisfaction,
    'salary_pct_of_market': salary_pct,
    'projects_completed_last_year': projects,
    'manager_interactions_last_quarter': mgr_interactions,
    'department': department,
    'job_level': job_level,
    'remote_status': remote_status,
    'manager_id': manager_id,
    'left_within_6mo': left_within_6mo
})

# --- 30 noise columns (TechCorp's HRIS exports many engagement telemetry metrics; most are noise) ---
# These columns are included because TechCorp's HR system really does export dozens of fields.
# They also make the selection-bias leak in Section 2.3 clearly visible.
NOISE_COLS = [f'engagement_metric_{j:02d}' for j in range(30)]
for col in NOISE_COLS:
    techcorp[col] = rng.normal(0.0, 1.0, size=n)

print('TechCorp Talent Analytics — first 5 rows (core columns):')
print(techcorp.iloc[:, :9].head())
print(f"\nShape (core + {len(NOISE_COLS)} noise engagement metrics): {techcorp.shape}")
print(f"Attrition rate: {techcorp['left_within_6mo'].mean():.1%}")
print("Categorical columns: ['department', 'job_level', 'remote_status', 'manager_id']")

**Reading the output:**

The dataframe prints 2,000 rows and 9 columns — 5 numeric features, 3 categorical features (stored as `object` dtype), and the binary target. The attrition rate is about 22%, i.e. the dataset is mildly imbalanced (`left_within_6mo = 1` is the minority class), which you now know how to handle: keep `stratify=y` in every split, and watch both ROC-AUC (threshold-free) and the confusion matrix.

The three categorical columns are visible immediately in `dtypes` — `object` dtype is pandas' way of saying *"text labels, not numbers."* Feeding these straight into `LogisticRegression` would fail with a "could not convert string to float" error. That's the problem `ColumnTransformer` solves.

---

### 2.2 ColumnTransformer — mixing numeric and categorical preprocessing

The rule is simple: every transform that depends on data statistics (mean, std, most-frequent category, one-hot vocabulary) **must be fit inside the pipeline** so cross-validation refits it on each fold's training portion only. A leak-free pipeline for TechCorp has two parallel branches, glued together by `ColumnTransformer`:

- **Numeric branch:** `StandardScaler` on `years_at_company`, `satisfaction_score`, `salary_pct_of_market`, `projects_completed_last_year`, `manager_interactions_last_quarter`.
- **Categorical branch:** `OneHotEncoder(handle_unknown='ignore')` on `department`, `job_level`, `remote_status`.

`handle_unknown='ignore'` is the safety net: if a prediction-time row has `department='QA'` (not seen during training), the encoder outputs all zeros for that row's department block instead of raising an exception. Without it, the pipeline would crash on any new category.

> 💡 **Gemini Prompt:** "Split TechCorp into X (features) and y (target `left_within_6mo`). Apply a stratified 60/20/20 train/val/test split with `random_state=RANDOM_SEED`. Build a `ColumnTransformer` with two branches: (a) `StandardScaler` on the 5 numeric columns, (b) `OneHotEncoder(handle_unknown='ignore')` on the 3 categorical columns. Wrap it in a `Pipeline` together with `LogisticRegression(random_state=RANDOM_SEED, max_iter=5000)`. Run 5-fold stratified `cross_val_score` on `X_train`, `y_train` with `scoring='roc_auc'`. Print the fold scores, mean, SD, and 95% CI using Student's t. Label the result `TechCorp baseline — leak-free pipeline`."


In [ ]:
# --- Stratified 60/20/20 split ---
CORE_NUM_COLS = ['years_at_company', 'satisfaction_score', 'salary_pct_of_market',
                 'projects_completed_last_year', 'manager_interactions_last_quarter']
NUM_COLS = CORE_NUM_COLS + NOISE_COLS  # feed all numeric columns into the pipeline
CAT_COLS_LOW_CARD = ['department', 'job_level', 'remote_status']
CAT_COLS = CAT_COLS_LOW_CARD + ['manager_id']  # manager_id is high-cardinality (80 levels)

X_tc = techcorp[NUM_COLS + CAT_COLS]
y_tc = techcorp['left_within_6mo']

X_tc_temp, X_tc_test, y_tc_temp, y_tc_test = train_test_split(
    X_tc, y_tc, test_size=0.20, random_state=RANDOM_SEED, stratify=y_tc
)
X_tc_train, X_tc_val, y_tc_train, y_tc_val = train_test_split(
    X_tc_temp, y_tc_temp, test_size=0.25,
    random_state=RANDOM_SEED, stratify=y_tc_temp
)
print(f'Train: {len(X_tc_train)} | Val: {len(X_tc_val)} | Test: {len(X_tc_test)} (locked)')

# --- ColumnTransformer + LogReg pipeline ---
preprocess = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)
])
tc_pipeline = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])

# --- 5-fold stratified CV with CI ---
cv_tc = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_scores_tc = cross_val_score(tc_pipeline, X_tc_train, y_tc_train,
                               cv=cv_tc, scoring='roc_auc')

mean_tc = cv_scores_tc.mean()
sd_tc = cv_scores_tc.std(ddof=1)
half_w_tc = t_crit * sd_tc / np.sqrt(k)

print('\nTechCorp baseline — leak-free pipeline')
print(f'Fold AUC: {np.round(cv_scores_tc, 4)}')
print(f'Mean AUC: {mean_tc:.4f}')
print(f'Std (k-1): {sd_tc:.4f}')
print(f'95% CI:   [{mean_tc - half_w_tc:.4f}, {mean_tc + half_w_tc:.4f}]')

**Reading the output:**

The TechCorp baseline pipeline prints five fold scores and the familiar mean ± 95% CI. Expect a mean ROC-AUC in the low- to mid-0.8s — not as sharp as MedScreen's 0.99, because HR data is noisier than medical imaging, but solidly better than the managers-guessing baseline (ROC-AUC = 0.50 by definition).

The crucial property of this pipeline is that **every statistic — standardization means, one-hot vocabularies — is computed inside each CV fold's training portion only**. Nothing the pipeline learns about column means or category levels is influenced by the held-out fold. That is what makes this number trustworthy.

Now watch what happens when an inexperienced pipeline leaks.

---

### 2.3 The intern's leaky pipeline — target encoding on full data

While you were away, an intern on the People Analytics team shipped a "first cut" of the TechCorp model. Their CV ROC-AUC came out *notably* higher than the baseline you just computed. They were celebrating. The CHRO wants to see the numbers.

The intern's reasoning: *"`manager_id` has 80 levels. One-hot encoding it would blow the feature space up to 100+ columns. The trick that Kaggle winners use is **target encoding** — replace each category with the mean of `y` within that category."* They applied the same trick to `department`, `job_level`, and `remote_status` while they were at it.

Here is exactly what the intern wrote:

```python
# Intern's approach
techcorp_te = techcorp.copy()
for col in CAT_COLS:  # includes manager_id
    techcorp_te[col + '_te'] = (
        techcorp_te.groupby(col)['left_within_6mo'].transform('mean')
    )
    techcorp_te = techcorp_te.drop(columns=[col])
```

Do you see it? The intern's `groupby(col)['left_within_6mo'].transform('mean')` computes each category's target mean using **every row's label** — training rows plus every row that will later land in a held-out fold. For `manager_id` (with ~25 employees per manager), each group mean is essentially an average of 25 labels — many of which will be *in the validation fold*. The encoded column is a near-perfect leak of the target.

The cell below reproduces the intern's inflated score, then re-runs the leak-free Section 2.2 baseline for comparison.

In [ ]:
# --- Intern's leaky pipeline: target encoding on the FULL dataset ---
techcorp_te = techcorp.copy()
for col in CAT_COLS:  # includes manager_id (80 levels)
    techcorp_te[col + '_te'] = (
        techcorp_te.groupby(col)['left_within_6mo'].transform('mean')
    )
    techcorp_te = techcorp_te.drop(columns=[col])

X_te = techcorp_te.drop(columns=['left_within_6mo'])
y_te = techcorp_te['left_within_6mo']

Xte_temp, Xte_test, yte_temp, yte_test = train_test_split(
    X_te, y_te, test_size=0.20, random_state=RANDOM_SEED, stratify=y_te
)
Xte_train, Xte_val, yte_train, yte_val = train_test_split(
    Xte_temp, yte_temp, test_size=0.25,
    random_state=RANDOM_SEED, stratify=yte_temp
)

leaky_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])
cv_leaky = cross_val_score(leaky_pipeline, Xte_train, yte_train,
                           cv=cv_tc, scoring='roc_auc')
print("Intern's leaky pipeline (target encoding fit on FULL data):")
print(f'  Fold AUC: {np.round(cv_leaky, 4)}')
print(f'  Mean AUC: {cv_leaky.mean():.4f}  ← inflated\n')

# --- Leak-free baseline from Section 2.2 (OHE of every categorical, no target encoding) ---
print('Leak-free baseline from Section 2.2 (OHE only, no target encoding):')
print(f'  Fold AUC: {np.round(cv_scores_tc, 4)}')
print(f'  Mean AUC: {mean_tc:.4f}  ← realistic')

delta_leak = cv_leaky.mean() - mean_tc
print(f'\n→ The leak inflated mean ROC-AUC by {delta_leak:+.4f}.')
print('  Target encoding on the full dataset hands the model the answer, in a single column.')

**Reading the output:**

Two side-by-side CV runs on the same data. The leaky version — where target encoding pre-computed each category's mean using every row's label, including rows later held out during CV — reports a mean ROC-AUC visibly higher than the leak-free version. The gap is the *size of the lie*. In production, the leak-free number is what you will actually see on new employees; the leaky number is fiction.

**Why this leak bites so hard on `manager_id`.** With 80 managers and 2,000 employees there are only about 25 rows per manager. The target mean within each manager is therefore an average of 25 labels — and when CV later holds out a fold, many of those 25 rows are *in* the held-out fold. The encoded column hands the classifier a direct summary of the target for exactly the rows it is supposed to predict.

**The fix is not "refit target encoding per fold" in this notebook.** Proper per-fold target encoding requires a custom transformer (`category_encoders.TargetEncoder` or a subclass of `BaseEstimator`/`TransformerMixin`) that integrates with sklearn's CV correctly. That is a deep-dive for later in the course. For today, the fix is the one-line rule you already know: **drop the leaky feature, fall back to `OneHotEncoder` inside the `Pipeline`** (exactly what the Section 2.2 baseline does).

**Rule of thumb:** if a feature is computed using *any* row's target, the only safe way to include it is a transformer that refits per CV fold. `OneHotEncoder` never touches `y`, so it is leak-safe by construction. Target encoders are not.

---

### 2.4 `FunctionTransformer` — domain features without leakage

Business stakeholders almost always want a feature that isn't in the raw data. At TechCorp the HR Business Partner is convinced that *ratios matter more than absolute values*: an employee whose `salary_pct_of_market` is 85 is more likely to leave than one at 130. That ratio already exists as a column, but the HRBP also wants `interactions_per_project = manager_interactions_last_quarter / (projects_completed_last_year + 1)` — a proxy for *management attention per unit of work*.

The amateur move: compute the ratio in a pandas `assign` before splitting and treat the new column as raw data. That works *for this example*, but if the feature depended on training statistics (a mean, a median, a category frequency) it would leak. The professional move is `FunctionTransformer`: wrap any pure function into a pipeline step so it runs fold-by-fold.

> 💡 **Gemini Prompt:** "Write a function `add_interactions_per_project(X)` that takes a pandas DataFrame with `manager_interactions_last_quarter` and `projects_completed_last_year`, returns a NumPy array with an added column `manager_interactions_last_quarter / (projects_completed_last_year + 1)`. Wrap it in a `FunctionTransformer(validate=False)`. Insert it as the first step in the pipeline (before the `ColumnTransformer`) and re-run 5-fold stratified CV on `X_tc_train`. Report the mean ROC-AUC and 95% CI, and compare to the baseline from Section 2.2."

In [ ]:
# --- FunctionTransformer to add interactions-per-project ratio ---
def add_interactions_per_project(X):
    X = X.copy()
    X['interactions_per_project'] = (
        X['manager_interactions_last_quarter'] /
        (X['projects_completed_last_year'] + 1)
    )
    return X

add_ratio = FunctionTransformer(add_interactions_per_project, validate=False)

NUM_COLS_RATIO = NUM_COLS + ['interactions_per_project']
preprocess_ratio = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS_RATIO),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)
])

tc_pipeline_ratio = Pipeline([
    ('add_ratio', add_ratio),
    ('prep', preprocess_ratio),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])

cv_tc_ratio = cross_val_score(tc_pipeline_ratio, X_tc_train, y_tc_train,
                              cv=cv_tc, scoring='roc_auc')

mean_ratio = cv_tc_ratio.mean()
sd_ratio = cv_tc_ratio.std(ddof=1)
half_w_ratio = t_crit * sd_ratio / np.sqrt(k)

print('TechCorp pipeline + FunctionTransformer(interactions_per_project)')
print(f'Mean AUC: {mean_ratio:.4f}')
print(f'95% CI:   [{mean_ratio - half_w_ratio:.4f}, {mean_ratio + half_w_ratio:.4f}]')

print('\nCompare to Section 2.2 baseline:')
print(f'  Baseline mean AUC: {mean_tc:.4f}  (CI [{mean_tc - half_w_tc:.4f}, {mean_tc + half_w_tc:.4f}])')
print(f'  +Ratio   mean AUC: {mean_ratio:.4f}  (CI [{mean_ratio - half_w_ratio:.4f}, {mean_ratio + half_w_ratio:.4f}])')

overlap_fe = not (mean_tc + half_w_tc < mean_ratio - half_w_ratio or
                  mean_ratio + half_w_ratio < mean_tc - half_w_tc)
print(f"\n→ CIs {'OVERLAP' if overlap_fe else 'do NOT overlap'}: "
      f"the ratio feature {'does not convincingly help' if overlap_fe else 'is a real edge'}.")

**Reading the output:**

Two pipelines, two CIs. If they overlap, the engineered ratio is not a statistically convincing lift — the HRBP's hypothesis is plausible but unproven on this sample, and you should leave the ratio out of the reported champion unless a separate argument (interpretability, stakeholder buy-in) keeps it in. If the CIs do not overlap, you have evidence that the ratio matters — quote it in the HR readout.

The pedagogical point is not whether the ratio helps on *this* synthetic dataset; it is the discipline: **every feature addition is tested with CV + CI overlap**, the same way NB08 tested Ridge vs OLS and C=1.0 vs C=0.01. You now have one tool (`FunctionTransformer`) that lets you add arbitrary domain features without leaking.

---

## 📝 PAUSE-AND-DO Exercise 2 (10 minutes)

**Task:** Find and fix a second leak — this one a different pattern.

Another teammate sends you the snippet below. They have heard about leakage and are careful: they are NOT doing target encoding. Instead they are using a *feature selector* to reduce the dimensionality of the one-hot-encoded feature matrix (pruning the 80 manager columns to only the ones that correlate with attrition). Their CV ROC-AUC came out a bit higher than your leak-free baseline. Read the code carefully before running.

```python
# Teammate's pipeline with univariate feature selection
X_all_dummies = pd.get_dummies(techcorp.drop(columns=['left_within_6mo']))
y_all = techcorp['left_within_6mo']

# Select the top-8 features by univariate correlation with the target — on ALL rows
selector = SelectKBest(f_classif, k=8)
X_selected = selector.fit_transform(X_all_dummies, y_all)

# Now split and CV on the pre-selected matrix
Xs_temp, Xs_test, ys_temp, ys_test = train_test_split(
    X_selected, y_all, test_size=0.20, random_state=RANDOM_SEED, stratify=y_all
)
Xs_train, Xs_val, ys_train, ys_val = train_test_split(
    Xs_temp, ys_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=ys_temp
)

pipe_sel = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])
cv_sel = cross_val_score(pipe_sel, Xs_train, ys_train, cv=cv_tc, scoring='roc_auc')
print(f'Pre-selected mean AUC: {cv_sel.mean():.4f}')
```

**Your job:**

1. Run the teammate's code and record the leaky mean.
2. Identify the leak in one sentence. *Hint:* `SelectKBest.fit(X, y)` reads `y` — does it do so on training rows only, or on every row?
3. Write a leak-free version by putting `SelectKBest` **inside** a `Pipeline` (chain it after the `ColumnTransformer` from Section 2.2, before the classifier). Run 5-fold stratified CV on `X_tc_train`, `y_tc_train`.
4. Print a final line: `Leak inflated mean ROC-AUC by +<delta>`.
5. Note: even though the inflation on this synthetic data may be small (a few thousandths to a few hundredths of a point), the *principle* is identical to Section 2.3. The Kaggle leaderboard does not reward "small" leaks; it punishes every leak equally.

---

> 💡 **Gemini Prompt:** "Run the teammate's `SelectKBest` snippet verbatim and record `cv_sel.mean()`. Then write a leak-free pipeline chaining Section 2.2's `ColumnTransformer` (`StandardScaler` on `NUM_COLS`, `OneHotEncoder(handle_unknown='ignore')` on `CAT_COLS`), `SelectKBest(f_classif, k=8)`, and `LogisticRegression(max_iter=5000)`. Run `cross_val_score(..., cv=cv_tc, scoring='roc_auc')` on `X_tc_train`, `y_tc_train`. Print both means and the delta."
>
> **After running, verify:**
> - The teammate's pre-selected mean AUC is higher than the leak-free version's mean AUC
> - Both pipelines use `k=8` selected features
> - The printed delta is positive (leak inflates the score)

In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance.
#
# What to do:
#   1. Reproduce the teammate's pre-select-then-split pipeline and record cv_sel.mean()
#   2. Build a leak-free pipeline: ColumnTransformer (Section 2.2) → SelectKBest(k=8) → LogisticRegression
#   3. Run cross_val_score on X_tc_train, y_tc_train with cv=cv_tc and scoring='roc_auc'
#   4. Print both means and the delta
#   5. In the analysis cell below, explain WHERE the leak is (one sentence)


### YOUR ANALYSIS:

**Question 1:** In one sentence, where is the leak in the teammate's code?  
[Your answer]

**Question 2:** By how much did the leak inflate the mean ROC-AUC? Is that inflation large enough to change the verdict of a CI-overlap comparison with a rival model?  
[Your answer]

**Question 3:** What is the one-line rule you would teach this teammate so this never happens again?  
[Your answer]

---

## 3. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **`GridSearchCV` is NB08's CV ritual, run on every point in a grid.** Every row of `cv_results_` has a mean, an SD, and (with two lines of `scipy.stats`) a 95% CI. Rank by mean, then break ties toward the simpler model using the CI-overlap rule.
2. **`RandomizedSearchCV` is the same idea for larger grids.** Sample `n_iter` points from distributions (`loguniform`, `uniform`) when an exhaustive grid would take too long. The output format is identical.
3. **`ColumnTransformer` is the bridge to real-world data.** Numeric columns need `StandardScaler`; categorical columns need `OneHotEncoder(handle_unknown='ignore')`. Both branches live inside one `ColumnTransformer`, which lives inside one `Pipeline`.
4. **`FunctionTransformer` embeds domain features without leaking.** Any pure function of a DataFrame can become a pipeline step. Test every new feature with the CI-overlap rule, not a one-shot validation comparison.
5. **Leakage hides inside `fit` calls on the full dataset.** Any step that takes `X` (for scaling, selection, encoding) or `y` (for feature selection, target encoding) must live inside the `Pipeline`. If the score looks suspiciously good, check for a `.fit(...)` outside `cross_val_score`.
6. **Both leaks in today's notebook inflated ROC-AUC by a non-trivial margin.** That gap is the *size of the lie* a leaky pipeline tells. On the Kaggle leaderboard, a leak of this size masks the real generalization gap and wrecks your final standing when the private test set is scored.

### Critical Rules:

> **"If a preprocessing step uses any data statistic, it belongs inside the `Pipeline`."**

> **"`cv_results_` is a stack of NB08 runs. Read it that way."**

> **"Break ties toward the simpler model — the CI-overlap rule is the rest of the course's decision primitive."**

### Next Steps:

- NB10 (Midterm): apply today's workflow to two business cases under time pressure.
- NB11–NB13 (trees, forests, gradient boosting): today's `GridSearchCV` is exactly how you will tune `max_depth`, `n_estimators`, `learning_rate`, and friends.
- Kaggle competition: the full pipeline you just built — `ColumnTransformer → FunctionTransformer → GridSearchCV → CI-overlap champion → refit on all training data → predict on test` — is the reference template for every submission.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in your code and analysis in both PAUSE-AND-DOs.
2. **Verify outputs**: Restart the runtime (`Runtime → Restart session`) and run all cells top to bottom. Every cell should execute without error.
3. **Download**: `File → Download → Download .ipynb` to save your completed notebook.
4. **Submit on Brightspace** in the Participation Assignments section.

### Grading:

- Exercise 1 complete with CI-overlap verdict: 50%
- Exercise 2 complete with leak diagnosis: 50%

---

## 4. Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2023). *An Introduction to Statistical Learning with Python* (ISLP), Ch. 5 (Resampling Methods) and Ch. 6 (Linear Model Selection and Regularization). Springer.
- Provost, F., & Fawcett, T. (2013). *Data Science for Business*. Chapters on overfitting, evaluation, and pipeline discipline.
- scikit-learn User Guide — [Tuning the hyper-parameters of an estimator](https://scikit-learn.org/stable/modules/grid_search.html)
- scikit-learn User Guide — [ColumnTransformer](https://scikit-learn.org/stable/modules/compose.html#columntransformer-for-heterogeneous-data)
- scikit-learn User Guide — [Common pitfalls in interpretation and evaluation](https://scikit-learn.org/stable/common_pitfalls.html)
- Kaufman, S., Rosset, S., & Perlich, C. (2012). *Leakage in Data Mining: Formulation, Detection, and Avoidance.*

---

<center>

Thank you!

</center>